In [ ]:
import pandas as pd
import numpy as np
import os


pi = 3.14159265359

maxval=1e9
minval=1e-9

In [ ]:
from dataloaders.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from models.mlp_encoder_model import *

In [ ]:
model=CreateModel_Slim((16,16,2))
model.summary()

In [ ]:
# get best weights file
pitch = '50x12P5'
batch_size = 5000
fingerprint = '34c2da80'
timeslices = 2
#files = os.listdir('/data/dajiang/smart-pixels/weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints'.format(pitch, batch_size, fingerprint, timeslices))
files = os.listdir('weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints'.format(pitch, batch_size, fingerprint, timeslices))

vlosses = [float(f.split("-v")[1].split(".hdf5")[0]) for f in files]
bestfile = files[np.argmin(vlosses)]
#model.load_weights('/data/dajiang/smart-pixels/weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints/'.format(pitch, batch_size, fingerprint, timeslices)+bestfile)
model.load_weights('weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints/'.format(pitch, batch_size, fingerprint, timeslices)+bestfile)

print('Best model: {}'.format(bestfile))

In [ ]:
weights_dir = 'weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints/'.format(pitch, batch_size, fingerprint, timeslices)
best_model_hdf5 = '{}/best_model-{}-bs{}-{}-{}t-mlp_SLIM-model_quantized.hdf5'.format(weights_dir, pitch, batch_size, fingerprint, timeslices)
best_model_keras = '{}/best_model-{}-bs{}-{}-{}t-mlp_SLIM-model_quantized.keras'.format(weights_dir, pitch, batch_size, fingerprint, timeslices)
best_model_weights_hdf5 = '{}/best_model_weights-{}-bs{}-{}-{}t-mlp_SLIM-model_quantized.hdf5'.format(weights_dir, pitch, batch_size, fingerprint, timeslices)
best_model_weights_keras = '{}/best_model_weights-{}-bs{}-{}-{}t-mlp_SLIM-model_quantized.keras'.format(weights_dir, pitch, batch_size, fingerprint, timeslices)
model_architecture_json = '{}/model_architecture-{}-bs{}-{}-{}t-mlp_SLIM-model_quantized.json'.format(weights_dir, pitch, batch_size, fingerprint, timeslices)

# Save (best) model information to file
model.save(best_model_hdf5)
model.save(best_model_keras)
model.save_weights(best_model_weights_hdf5)
model.save_weights(best_model_weights_keras)
model_json = model.to_json()
with open(model_architecture_json, "w") as json_file:
    json_file.write(model_json)

In [ ]:
# load in the test set
test_generator = OptimizedDataGenerator(
    #load_from_tfrecords_dir = '/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/TFR_files/{}t/TFR_val_contained_digitize-manual_mlp-SLIM'.format(timeslices),
    load_from_tfrecords_dir = '/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/TFR_files/{}t/TFR_val_contained_digitize-manual_mlp-SLIM'.format(timeslices),
    quantize = False # False for soft quantizer and manually quantized inputs
)

In [ ]:
# predicts test data
p_test = model.predict(test_generator)

complete_truth = None
for _, y in test_generator:
    if complete_truth is None:
        complete_truth = y
    else:
        complete_truth = np.concatenate((complete_truth, y), axis=0)

# creates df with all predicted values and matrix elements - 4 predictions, all 10 unique matrix elements
df = pd.DataFrame(p_test,columns=['x','y','cotB'])

# stores all true values in same matrix as xtrue, ytrue, etc.
df['xtrue'] = complete_truth[:,0]
df['ytrue'] = complete_truth[:,1]
df['cotBtrue'] = complete_truth[:,2]

# calculates residuals for x, y, cotA, cotB
df['residualsX'] = df['xtrue'] - df['x']
df['residualsY'] = df['ytrue'] - df['y']
df['residualsB'] = df['cotBtrue'] - df['cotB']

# stores results as parquet
#df.to_parquet("/home/dajiang/smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5/{}t-mlp_SLIM-manual_input_digitization-vars.parquet".format(timeslices))
df.to_parquet("/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/TFR_files/{}t-mlp_SLIM-manual_input_digitization-vars.parquet".format(timeslices))

In [ ]:
# Create a numpy array that contains the data and labels information
# X is the 2-timeslices data
# y is the labels data ['x-midplane','y-midplane','?']
X_val_all = []
y_val_all = []

num_batches = test_generator.__len__() # The total number of batches for the validation dataset
#val_num_batches = 1

for i_batch in range(num_batches): # Loop over all batches
    X_val, y_val = test_generator.__getitem__(i_batch)
    X_val = X_val.numpy()
    y_val = y_val.numpy()
    X_val_all.append(X_val)
    y_val_all.append(y_val)

X_val_all = np.array(np.concatenate(X_val_all))
y_val_all = np.array(np.concatenate(y_val_all))

np.save(f"npy/X_val.npy", X_val_all)
np.save(f"npy/y_val.npy", y_val_all)

In [ ]:
import numpy as np
unique_vals = set(np.ravel(X_val_all).tolist())


In [ ]:
print(unique_vals)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# If X may contain NaN/inf
X_clean = np.asarray(X_val_all)
X_clean = X_clean[np.isfinite(X_clean)]

plt.figure()
plt.hist(X_clean, bins='auto', edgecolor='black')  # try bins=30 for fixed bins
plt.xlabel('X')
plt.ylabel('Count')
plt.title('Data distribution')
plt.tight_layout()
plt.show()
